# GreenFinance-Scorer — Étape 4 : validation autonome du pipeline d'extraction

Ce notebook répond à une seule question : le pipeline extrait-il des données ESG/carbone fiables,
avec des citations de page exactes, à partir de rapports PDF réels ?

Aucune authentification, aucune API, aucun branchement à la base de données de l'Étape 3 — un notebook
isolé, avec un index de recherche sémantique en mémoire (pas pgvector), volontairement découplé du schéma
applicatif. C'est aussi la première version du notebook de démonstration (Livrable 6).

**Critères de fiabilité (non négociables) :**
- ≥ 90 % des valeurs numériques extraites correspondent à la vérité terrain (tolérance ± 2 %)
- ≥ 85 % des citations de page sont exactes
- 0 % d'hallucination : une donnée absente du rapport doit toujours être signalée comme non disponible

## Prompt 4.1 — Préparation du corpus pilote et de la vérité terrain

Charge `data_test/ground_truth.yaml` et vérifie que chaque PDF référencé existe bien
sous `storage/pilot_corpus/`.

In [ ]:
import re
from pathlib import Path

import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GROUND_TRUTH_PATH = PROJECT_ROOT / "data_test" / "ground_truth.yaml"
PILOT_CORPUS_DIR = PROJECT_ROOT / "storage" / "pilot_corpus"

with open(GROUND_TRUTH_PATH, "r", encoding="utf-8") as f:
    ground_truth = yaml.safe_load(f)

entreprises = ground_truth["entreprises"]


def parse_pages_attendues(value):
    """Normalise page_attendue (entier, ou plage texte "82-83") en un frozenset de numeros de page."""
    if isinstance(value, int):
        return frozenset({value})
    match = re.match(r"^\s*(\d+)\s*-\s*(\d+)\s*$", str(value))
    if match:
        start, end = int(match.group(1)), int(match.group(2))
        return frozenset(range(start, end + 1))
    return frozenset({int(value)})


for entreprise in entreprises:
    for indicateur in entreprise["indicateurs"]:
        indicateur["pages_attendues"] = parse_pages_attendues(indicateur["page_attendue"])

print(f"{len(entreprises)} entreprises chargees depuis {GROUND_TRUTH_PATH.relative_to(PROJECT_ROOT)}")

total_indicateurs = 0
missing_files = []
for entreprise in entreprises:
    doc_path = PROJECT_ROOT / entreprise["document"]
    n = len(entreprise["indicateurs"])
    total_indicateurs += n
    status = "OK" if doc_path.exists() else "MANQUANT"
    if not doc_path.exists():
        missing_files.append(str(doc_path))
    print(f"  {entreprise['nom']:15s} {n} indicateurs   PDF: {status}")

print(f"\nTotal indicateurs (verite terrain) : {total_indicateurs}")
if missing_files:
    raise FileNotFoundError(f"PDF manquants : {missing_files}")
print("Tous les PDF du corpus pilote sont presents.")


## Prompt 4.2 — Lecture et structuration (Docling + PaddleOCR)

Pour chaque PDF du corpus pilote : nombre de pages, nombre de tableaux structurés, déclenchement (ou
non) de l'OCR, temps de traitement.

In [ ]:
import os

# Docling utilise des modeles PyTorch (TableFormer, layout) qui tentent par defaut de se compiler
# via torch.compile (backend inductor) pour accelerer l'inference. Cette compilation a besoin d'un
# compilateur C++ (cl.exe sous Windows) absent de cet environnement : sans le desactiver, chaque appel
# modele echoue et retente en boucle sur des centaines de pages, jusqu'a saturer la memoire et tuer le
# kernel. torch.compile n'est qu'une optimisation de vitesse (inutile pour ce notebook d'un seul run) ;
# le desactiver evite d'avoir a installer une chaine de compilation C++ pour un gain qu'on n'exploite
# qu'une fois. Doit etre positionne AVANT tout import de torch/docling.
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import time

import pymupdf
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = True

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

conversion_results = {}  # nom -> {document, n_pages, n_tables, n_pages_sans_texte_natif, elapsed_s}

for entreprise in entreprises:
    nom = entreprise["nom"]
    doc_path = PROJECT_ROOT / entreprise["document"]
    print(f"Conversion Docling : {nom} ({doc_path.name}) ...")

    start = time.time()
    result = converter.convert(str(doc_path))
    elapsed = time.time() - start
    doc = result.document

    # Pages sans couche texte native (via PyMuPDF) : candidates a l'OCR, independamment
    # de la decision interne du pipeline Docling -- mesure objective et reproductible.
    pdf = pymupdf.open(str(doc_path))
    n_pages_sans_texte = sum(1 for page in pdf if len((page.get_text() or "").strip()) < 20)
    pdf.close()

    conversion_results[nom] = {
        "document": doc,
        "n_pages": len(doc.pages),
        "n_tables": len(doc.tables),
        "n_pages_sans_texte_natif": n_pages_sans_texte,
        "elapsed_s": round(elapsed, 1),
    }
    print(
        f"  -> {len(doc.pages)} pages, {len(doc.tables)} tableaux structures, "
        f"{n_pages_sans_texte} page(s) sans texte natif (OCR necessaire), {elapsed:.1f}s"
    )

print(f"\n{'Entreprise':15s} {'Pages':>6s} {'Tableaux':>9s} {'OCR necessaire':>15s} {'Temps (s)':>10s}")
for nom, r in conversion_results.items():
    print(
        f"{nom:15s} {r['n_pages']:6d} {r['n_tables']:9d} "
        f"{r['n_pages_sans_texte_natif']:15d} {r['elapsed_s']:10.1f}"
    )


## Prompt 4.3 — Recherche sémantique (bge-m3, en mémoire)

Deux requêtes fixes (identiques pour tous les rapports, jamais générées depuis le code de
l'indicateur — voir `DEUX_REQUETES_FIXES`), cherchées **exhaustivement** (`k=index.ntotal`, donc un
rang exact sur tout le rapport, pas une approximation tronquée), puis fusionnées par page physique
(meilleur score conservé). `retrieval_recall_at_8` (alias `hit_top8`) mesure si la page attendue de
chaque indicateur figure dans les 8 mieux classées.

**Le top-8 est un benchmark de récupération documentaire et ne limite pas l'analyse intégrale des
rapports dans le système métier** — le Prompt 4.4+ traite l'intégralité du contenu extrait, jamais
seulement un top-8.


In [ ]:
import sys

import faiss
import numpy as np
from FlagEmbedding import BGEM3FlagModel

print("Chargement du modele bge-m3 (peut telecharger plusieurs Go au premier appel)...")
embed_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)

# Protocole (deuxieme correction) : deux requetes fixes, identiques pour tous les rapports, cherchees
# EXHAUSTIVEMENT (k=index.ntotal) puis consolidees par page -- rang exact, pas une approximation
# tronquee. Voir notebooks/_prompt_4_3_indexation.py pour la justification complete de chaque requete.
DEUX_REQUETES_FIXES = [
    (
        "Scope 1, Scope 2 location-based, Scope 2 market-based and Scope 3 greenhouse gas emissions "
        "in tonnes of CO2 equivalent (tCO2e)"
    ),
    "Greenhouse gas emissions intensity in grams of CO2 equivalent per kilowatt-hour (gCO2e/kWh)",
]
TOP_PAGES = 8  # troncature du classement complet pour le benchmark retrieval_recall_at_8


def iter_page_chunks(doc, max_chars=1200):
    """Reconstitue, a partir d'un DoclingDocument, des chunks de texte associes a leur page d'origine.

    Une erreur d'export_to_markdown est journalisee (page + erreur), jamais masquee silencieusement.
    """
    buffers = {}  # page_no -> list[str]
    for item in list(getattr(doc, "texts", [])) + list(getattr(doc, "tables", [])):
        prov = getattr(item, "prov", None)
        if not prov:
            continue
        page_no = prov[0].page_no
        content = None
        if getattr(item, "text", None):
            content = item.text
        elif hasattr(item, "export_to_markdown"):
            try:
                content = item.export_to_markdown(doc)
            except Exception as exc:  # noqa: BLE001 — journalisé, jamais avalé (voir docstring)
                print(
                    f"AVERTISSEMENT: export_to_markdown a echoue page {page_no} "
                    f"({type(exc).__name__}: {exc}) -- contenu de cet element perdu",
                    file=sys.stderr,
                )
                content = None
        if not content:
            continue
        buffers.setdefault(page_no, []).append(content)

    chunks = []
    for page_no, parts in buffers.items():
        text = "\n".join(parts)
        for i in range(0, len(text), max_chars):
            piece = text[i : i + max_chars].strip()
            if piece:
                chunks.append({"page": page_no, "text": piece})
    return chunks


search_indexes = {}  # nom -> {"chunks": [...], "index": faiss.Index}

for nom, r in conversion_results.items():
    chunks = iter_page_chunks(r["document"])
    if not chunks:
        print(f"{nom}: aucun chunk extrait, indexation ignoree")
        continue
    embeddings = embed_model.encode([c["text"] for c in chunks], batch_size=8)["dense_vecs"]
    embeddings = np.asarray(embeddings, dtype="float32")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    search_indexes[nom] = {"chunks": chunks, "index": index}
    print(f"{nom}: {len(chunks)} chunks indexes")


def search_all_chunks(nom, query):
    """Recherche EXHAUSTIVE (k=ntotal) : IndexFlatIP calcule deja la similarite avec tous les vecteurs
    en interne quel que soit k, donc k=ntotal ne coute rien de plus qu'un k tronque -- et c'est la
    seule facon d'obtenir un rang exact, pas une approximation bornee par un k arbitraire."""
    entry = search_indexes[nom]
    q_emb = embed_model.encode([query])["dense_vecs"]
    q_emb = np.asarray(q_emb, dtype="float32")
    faiss.normalize_L2(q_emb)
    k = entry["index"].ntotal
    scores, idxs = entry["index"].search(q_emb, k)
    return [
        (entry["chunks"][i], float(score))
        for i, score in zip(idxs[0], scores[0])
        if i != -1
    ]


def consolidate_by_page(chunk_score_pairs):
    """Regroupe par page physique -- conserve le MEILLEUR score par page -- trie decroissant.
    Avec une recherche exhaustive en entree, ce classement couvre TOUTES les pages du rapport."""
    best_by_page = {}
    for chunk, score in chunk_score_pairs:
        page = chunk["page"]
        if page not in best_by_page or score > best_by_page[page]:
            best_by_page[page] = score
    return sorted(best_by_page.items(), key=lambda item: item[1], reverse=True)


# retrieval_recall_at_8 (alias hit_top8) est UNIQUEMENT un benchmark technique de qualite de
# recherche documentaire : au moins une page attendue figure-t-elle dans les 8 mieux classees ?
# Le top-8 est un benchmark de recuperation documentaire et ne limite pas l'analyse integrale des
# rapports dans le systeme metier -- le Prompt 4.4+ traite l'integralite du contenu extrait.
recall_hits = 0
recall_total = 0

for entreprise in entreprises:
    nom = entreprise["nom"]
    if nom not in search_indexes:
        continue

    chunk_scores = []
    for requete in DEUX_REQUETES_FIXES:
        chunk_scores.extend(search_all_chunks(nom, requete))

    classement_complet = consolidate_by_page(chunk_scores)
    rang_par_page = {page: rang + 1 for rang, (page, _score) in enumerate(classement_complet)}

    hits = 0
    n = len(entreprise["indicateurs"])
    for indicateur in entreprise["indicateurs"]:
        pages_attendues = indicateur["pages_attendues"]
        pages_retrouvees = [p for p in pages_attendues if p in rang_par_page]
        rang_trouve = min((rang_par_page[p] for p in pages_retrouvees), default=None)
        hit_top8 = rang_trouve is not None and rang_trouve <= TOP_PAGES
        if hit_top8:
            hits += 1
            recall_hits += 1
        recall_total += 1
    print(f"{nom:15s} retrieval_recall_at_8 (benchmark recherche) : {hits}/{n} ({hits / n:.0%})")

print(
    f"\nretrieval_recall_at_8 global (benchmark technique, pas une limite du pipeline metier) : "
    f"{recall_hits}/{recall_total} ({recall_hits / recall_total:.1%})"
)


## Prompt 4.4 — Extraction structurée via Claude, avec citation de page

Appel Claude sur les `CONTEXT_TOP_PAGES` (20) meilleures pages consolidées des deux requêtes fixes du Prompt 4.3 — pas seulement le top-8 du benchmark de recherche, qui ne mesure que la qualité de récupération, pas la fenêtre d'extraction (voir `_prompt_4_4_extraction.py`). Réponse contrainte par un schéma Pydantic strict (`valeur`, `unité`, `page_source`, `trouvé`) ; règle explicite de ne jamais inventer une valeur ni une page absente du contexte fourni. Résultats bruts consignés dans `data_test/extraction_results.json`.


In [ ]:
import json
import os

import anthropic
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(PROJECT_ROOT / ".env")

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
CLAUDE_MODEL = "claude-sonnet-5"

# Fenetre d'extraction -- distincte de TOP_PAGES=8 (benchmark retrieval_recall_at_8 du Prompt 4.3, une
# mesure de qualite de recherche, pas la fenetre a donner a Claude). Choisie pour couvrir avec marge le
# pire rang observe au 4.3 (rang 10, Microsoft) tout en restant tres inferieure au nombre de pages des
# gros rapports (218 pour Orsted). A reevaluer seulement apres les resultats complets Microsoft/Orsted --
# jamais ajustee retroactivement pour ameliorer un score.
CONTEXT_TOP_PAGES = 20


class IndicateurExtrait(BaseModel):
    code: str
    valeur: float | None = Field(default=None, description="Valeur numerique trouvee, null si absente des extraits fournis")
    unite: str | None = Field(default=None, description="Unite de la valeur, telle qu'ecrite dans le document")
    page_source: int | None = Field(default=None, description="Numero de page physique (indique par --- Page N ---) ou la valeur a ete trouvee")
    trouve: bool = Field(description="False si l'indicateur n'apparait pas explicitement dans les extraits fournis")


class ExtractionEntreprise(BaseModel):
    entreprise: str
    indicateurs: list[IndicateurExtrait]


EXTRACTION_TOOL = {
    "name": "extraction_indicateurs",
    "description": "Extraction structuree des indicateurs ESG/carbone demandes, a partir des extraits de rapport fournis.",
    "input_schema": ExtractionEntreprise.model_json_schema(),
}


def build_context(nom):
    """Assemble le texte COMPLET (tous les chunks, pas seulement le meilleur par page) des
    CONTEXT_TOP_PAGES meilleures pages consolidees des deux requetes fixes du Prompt 4.3 -- une seule
    fois pour tous les indicateurs de l'entreprise, jamais une requete/un contexte different par
    indicateur (c'est exactement le defaut que le Prompt 4.3 a corrige)."""
    chunk_scores = []
    for requete in DEUX_REQUETES_FIXES:
        chunk_scores.extend(search_all_chunks(nom, requete))

    classement = consolidate_by_page(chunk_scores)
    top_pages = {page for page, _score in classement[:CONTEXT_TOP_PAGES]}

    pages_text = {}
    for chunk in search_indexes[nom]["chunks"]:
        if chunk["page"] in top_pages:
            pages_text.setdefault(chunk["page"], []).append(chunk["text"])

    ordered_pages = sorted(pages_text)
    context = "\n\n".join(f"--- Page {p} ---\n" + "\n".join(pages_text[p]) for p in ordered_pages)
    return context, ordered_pages


extraction_results = {}
# Pages effectivement fournies a Claude par entreprise -- reutilise au Prompt 4.5 pour distinguer une
# hallucination (valeur affirmee sans qu'aucune page attendue n'ait ete presente dans ce contexte) d'un
# simple echec de recherche documentaire (page attendue jamais dans les CONTEXT_TOP_PAGES).
pages_used_by_entreprise = {}

for entreprise in entreprises:
    nom = entreprise["nom"]
    if nom not in search_indexes:
        print(f"{nom}: pas d'index, extraction ignoree")
        continue

    context, pages_used = build_context(nom)
    pages_used_by_entreprise[nom] = set(pages_used)
    codes = [i["code"] for i in entreprise["indicateurs"]]
    print(f"{nom}: contexte assemble sur {len(pages_used)} pages ({CONTEXT_TOP_PAGES} demandees)")

    prompt = f"""Tu es un extracteur de donnees ESG/carbone. Voici des extraits d'un rapport d'entreprise
({nom}), avec le numero de page physique indique avant chaque extrait.

Pour CHACUN des indicateurs suivants, trouve sa valeur numerique la plus recente dans les extraits fournis :
{json.dumps(codes, ensure_ascii=False)}

Regles strictes :
- Si un indicateur n'apparait pas explicitement dans les extraits fournis, "trouve" doit etre false et
  "valeur" doit etre null. Ne jamais inventer une valeur ni une page.
- "page_source" doit etre le numero de page (indique par "--- Page N ---") ou tu as trouve la valeur.
- Reponds pour chacun des {len(codes)} indicateurs demandes, dans le meme ordre.

Extraits du rapport :
{context}
"""

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4096,
        tools=[EXTRACTION_TOOL],
        tool_choice={"type": "tool", "name": "extraction_indicateurs"},
        messages=[{"role": "user", "content": prompt}],
    )
    tool_use = next(b for b in response.content if b.type == "tool_use")
    result = tool_use.input
    extraction_results[nom] = result
    print(f"{nom}: {len(result.get('indicateurs', []))} indicateurs extraits (contexte : {len(pages_used)} pages)")

RESULTS_PATH = PROJECT_ROOT / "data_test" / "extraction_results.json"
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(extraction_results, f, ensure_ascii=False, indent=2)
print(f"\nResultats bruts ecrits dans {RESULTS_PATH.relative_to(PROJECT_ROOT)}")


## Prompt 4.5 — Rapport de précision automatisé

Les trois métriques de fiabilité, par entreprise (pas seulement une moyenne globale).

In [ ]:
def within_tolerance(expected, actual, tol=0.02):
    if actual is None:
        return False
    if expected == 0:
        return actual == 0
    return abs(actual - expected) / abs(expected) <= tol


report_rows = []

for entreprise in entreprises:
    nom = entreprise["nom"]
    extracted = extraction_results.get(nom)
    if not extracted:
        continue

    extracted_by_code = {i["code"]: i for i in extracted.get("indicateurs", [])}
    pages_used = pages_used_by_entreprise.get(nom, set())

    n = len(entreprise["indicateurs"])
    correct_values = 0
    correct_pages = 0
    hallucinations = 0
    details = []

    for gt in entreprise["indicateurs"]:
        found = extracted_by_code.get(gt["code"])
        page_was_available = bool(pages_used & gt["pages_attendues"])

        value_ok = False
        page_ok = False
        is_hallucination = False

        if found is not None and found.get("trouve") and found.get("valeur") is not None:
            value_ok = within_tolerance(gt["valeur_attendue"], found["valeur"])
            page_ok = found.get("page_source") in gt["pages_attendues"]
            # Hallucination : Claude affirme une valeur alors qu'aucune des pages attendues n'a jamais
            # ete fournie dans son contexte -- la donnee ne pouvait pas etre legitimement trouvee.
            if not page_was_available:
                is_hallucination = True

        correct_values += int(value_ok)
        correct_pages += int(page_ok)
        hallucinations += int(is_hallucination)

        if not (value_ok and page_ok) or is_hallucination:
            details.append(
                {
                    "code": gt["code"],
                    "attendu": gt["valeur_attendue"],
                    "trouve": found.get("valeur") if found else None,
                    "page_attendue": gt["page_attendue"],
                    "page_trouvee": found.get("page_source") if found else None,
                    "page_disponible_dans_contexte": page_was_available,
                    "valeur_correcte": value_ok,
                    "page_correcte": page_ok,
                    "hallucination": is_hallucination,
                }
            )

    report_rows.append(
        {
            "entreprise": nom,
            "n_indicateurs": n,
            "valeurs_correctes": correct_values,
            "pages_correctes": correct_pages,
            "hallucinations": hallucinations,
            "taux_valeurs": correct_values / n,
            "taux_pages": correct_pages / n,
            "details_echecs": details,
        }
    )
    print(
        f"{nom:15s} valeurs: {correct_values}/{n} ({correct_values / n:.0%})   "
        f"pages: {correct_pages}/{n} ({correct_pages / n:.0%})   hallucinations: {hallucinations}"
    )

total_n = sum(r["n_indicateurs"] for r in report_rows)
total_valeurs = sum(r["valeurs_correctes"] for r in report_rows)
total_pages = sum(r["pages_correctes"] for r in report_rows)
total_hallu = sum(r["hallucinations"] for r in report_rows)

print(
    f"\n{'GLOBAL':15s} valeurs: {total_valeurs}/{total_n} ({total_valeurs / total_n:.1%})   "
    f"pages: {total_pages}/{total_n} ({total_pages / total_n:.1%})   hallucinations: {total_hallu}"
)

PRECISION_REPORT_PATH = PROJECT_ROOT / "data_test" / "precision_report.json"
with open(PRECISION_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "par_entreprise": report_rows,
            "global": {
                "taux_valeurs": total_valeurs / total_n,
                "taux_pages": total_pages / total_n,
                "hallucinations": total_hallu,
                "n_indicateurs": total_n,
            },
        },
        f,
        ensure_ascii=False,
        indent=2,
    )
print(f"Rapport ecrit dans {PRECISION_REPORT_PATH.relative_to(PROJECT_ROOT)}")


## Prompt 4.6 — Décision de fiabilité et finalisation

Verdict GO / NO-GO documenté, avec les cas d'échec identifiés et la décision qui en découle.

In [ ]:
from IPython.display import Markdown, display

with open(PRECISION_REPORT_PATH, "r", encoding="utf-8") as f:
    precision = json.load(f)

SEUIL_VALEURS = 0.90
SEUIL_PAGES = 0.85
SEUIL_HALLUCINATIONS = 0

g = precision["global"]
ok_valeurs = g["taux_valeurs"] >= SEUIL_VALEURS
ok_pages = g["taux_pages"] >= SEUIL_PAGES
ok_hallu = g["hallucinations"] <= SEUIL_HALLUCINATIONS
go = ok_valeurs and ok_pages and ok_hallu

lines = [
    "| Critere | Seuil | Resultat | Statut |",
    "|---|---|---|---|",
    f"| Valeurs correctes (+/-2%) | >= {SEUIL_VALEURS:.0%} | {g['taux_valeurs']:.1%} | {'OK' if ok_valeurs else 'ECHEC'} |",
    f"| Citations de page exactes | >= {SEUIL_PAGES:.0%} | {g['taux_pages']:.1%} | {'OK' if ok_pages else 'ECHEC'} |",
    f"| Hallucinations | = 0 | {g['hallucinations']} | {'OK' if ok_hallu else 'ECHEC'} |",
    "",
    f"## Verdict : {'GO' if go else 'NO-GO'}",
    "",
]

echecs = [row for row in precision["par_entreprise"] if row.get("details_echecs")]
if echecs:
    lines.append("### Cas d'echec identifies\n")
    for row in echecs:
        lines.append(f"**{row['entreprise']}**")
        for d in row["details_echecs"]:
            nature = []
            if d["hallucination"]:
                nature.append("HALLUCINATION")
            if not d["valeur_correcte"]:
                nature.append("valeur incorrecte")
            if not d["page_correcte"]:
                nature.append("page incorrecte")
            lines.append(
                f"- `{d['code']}` ({', '.join(nature)}) : attendu {d['attendu']} p.{d['page_attendue']}, "
                f"obtenu {d['trouve']} p.{d['page_trouvee']}"
            )
        lines.append("")
else:
    lines.append("Aucun cas d'echec : tous les indicateurs ont ete extraits avec la valeur et la page attendues.")

lines.append("")
if go:
    lines.append(
        "**Decision : GO.** Les trois criteres de fiabilite sont atteints sur le corpus pilote "
        f"({len(precision['par_entreprise'])} entreprises, {g['n_indicateurs']} indicateurs). "
        "Le pipeline peut etre porte dans `app/ingestion/` a l'Etape 5."
    )
else:
    manque = []
    if not ok_valeurs:
        manque.append("la precision des valeurs")
    if not ok_pages:
        manque.append("la precision des citations de page")
    if not ok_hallu:
        manque.append("l'absence d'hallucination")
    lines.append(
        f"**Decision : NO-GO.** Le pipeline n'atteint pas le seuil requis sur : {', '.join(manque)}. "
        "Voir les cas d'echec ci-dessus avant de reprendre l'integration a l'Etape 5."
    )

display(Markdown("\n".join(lines)))
